# VIX Portfolio Insurance — Hedge Analysis

This notebook connects to your IBKR portfolio (or uses a demo portfolio), fetches delayed VIX options data from free sources, and provides a complete portfolio insurance analysis using VIX call options.

**Sections:**
1. Setup & IBKR Connection
2. Portfolio Snapshot
3. Portfolio Risk Profile
4. VIX Options Chain
5. VIX Call Analytics & Comparison
6. Scenario Analysis — Drawdown Modelling
7. Recommendations & Rollover Strategy

## 1. Setup & IBKR Connection

In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, HTML
import yfinance as yf
from pathlib import Path

# ibkr_eda modules
from ibkr_eda.hedging import VIXData, ScenarioEngine, HedgeAdvisor, estimate_portfolio_beta
from ibkr_eda.hedging.config import (
    DEMO_PORTFOLIO, STRESS_EVENTS, VIX_MULTIPLIER,
    PLOTLY_LAYOUT, COLORS, DATA_DIR,
)
from ibkr_eda.dashboard_v2.analytics.drawdown import compute_underwater, compute_top_drawdowns
from ibkr_eda.dashboard_v2.analytics.returns_metrics import compute_metrics_table
from ibkr_eda.dashboard_v2.analytics.rolling import rolling_beta, rolling_volatility
from ibkr_eda.dashboard_v2.analytics.correlation import compute_correlation_matrix
from ibkr_eda.dashboard_v2.analytics.risk_contribution import compute_risk_contribution

# Ensure data directory exists
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Imports OK ✓")

Imports OK ✓


In [ ]:
# --- Connect to IBKR or fall back to demo mode ---

LIVE_MODE = True
ib = None

try:
    from ibkr_eda import IBKR
    ib = await IBKR.create_async()
    status = ib.status()
    LIVE_MODE = True
    display(HTML(
        '<div style="padding:10px;background:#0a3d0a;border-left:4px solid #00ff88;color:#00ff88">'
        f'<b>Connected to IBKR</b> — {status}</div>'
    ))
except Exception as e:
    LIVE_MODE = False
    display(HTML(
        '<div style="padding:10px;background:#3d0a0a;border-left:4px solid #ff6b6b;color:#ff6b6b">'
        '<b>TWS not available</b> — running in <b>demo mode</b> with sample portfolio.<br>'
        f'<small>Error: {e}</small><br><br>'
        '<em>To connect: Start TWS/IB Gateway → Enable API on port 4001 (live) or 4002 (paper) → '
        'Check your <code>.env</code> file.</em></div>'
    ))

Error 300, reqId 77: Can't find EId with tickerId:77
Error 300, reqId 120: Can't find EId with tickerId:120
Error 300, reqId 121: Can't find EId with tickerId:121
Error 300, reqId 122: Can't find EId with tickerId:122
Error 300, reqId 123: Can't find EId with tickerId:123
Error 300, reqId 124: Can't find EId with tickerId:124
Error 300, reqId 125: Can't find EId with tickerId:125
Error 300, reqId 126: Can't find EId with tickerId:126
Error 300, reqId 127: Can't find EId with tickerId:127
Error 300, reqId 128: Can't find EId with tickerId:128
Error 300, reqId 129: Can't find EId with tickerId:129
Error 300, reqId 130: Can't find EId with tickerId:130
Error 300, reqId 131: Can't find EId with tickerId:131
Error 300, reqId 132: Can't find EId with tickerId:132
Error 300, reqId 133: Can't find EId with tickerId:133
Error 300, reqId 134: Can't find EId with tickerId:134
Error 300, reqId 135: Can't find EId with tickerId:135
Error 300, reqId 136: Can't find EId with tickerId:136
Error 300, r

In [3]:

# --- Helper: fetch current prices via IBKR snapshot (live) or yfinance (demo/fallback) ---
from ibkr_eda.dashboard_v2.data.price_fetcher import ibkr_to_yfinance

# LSE exchanges return prices in GBX (pence) — must divide by 100 to get GBP (pounds)
_LSE_EXCHANGES = {"LSE", "LSEETF", "TRQXUK", "TRWBUKETF"}
_GBP_PENCE_YF_SUFFIXES = {".L"}


async def fetch_current_prices(
    tickers: list[str],
    exchange_map: dict[str, str] | None = None,
    ib=None,
    conid_map: dict[str, int] | None = None,
) -> pd.Series:
    """Fetch last price for a list of IBKR tickers.

    In LIVE_MODE (ib + conid_map provided): queries IBKR snapshot directly,
    giving native-currency prices without yfinance symbol mapping overhead.
    Otherwise: falls back to yfinance with disk cache.

    LSE stocks (exchange in _LSE_EXCHANGES) are returned by both IBKR
    snapshot and yfinance in GBX (pence). Prices are divided by 100 here
    so all returned values are in the contract's stated currency (GBP).

    Parameters
    ----------
    tickers : IBKR symbols
    exchange_map : optional dict mapping IBKR symbol → IBKR exchange.
        Used to identify LSE stocks for pence → pound conversion.
    ib : IBKR client instance (live mode only)
    conid_map : dict mapping IBKR symbol → contract_id (live mode only)
    """
    em = exchange_map or {}
    lse_tickers = {t for t, exch in em.items() if exch in _LSE_EXCHANGES}

    # --- IBKR live path: native currency, no symbol translation needed ---
    if ib is not None and conid_map:
        conids = [int(conid_map[t]) for t in tickers if t in conid_map and conid_map[t]]
        if conids:
            snap = await ib.snapshot.get_async(conids)
            if not snap.empty:
                # prefer last price; fall back to close when market is closed
                snap["price"] = snap["last"].combine_first(snap["close"])
                # Drop zeros — TWS returns 0 (not NaN) for stale/unavailable prices
                # (e.g. ASX stocks outside Australian trading hours).
                # Those tickers will fall through to the yfinance path below.
                snap = snap[snap["price"].fillna(0) > 0]
                prices = snap.set_index("symbol")["price"].dropna().rename("price").copy()
                # IBKR snapshot returns LSE prices in GBX (pence) → convert to GBP
                for t in lse_tickers:
                    if t in prices.index:
                        prices[t] = prices[t] / 100.0
                # Only return if ALL requested tickers have a valid price.
                # If some are missing (zero / unavailable), fall through to yfinance
                # so the full set is fetched consistently from one source.
                if not prices.empty and set(tickers).issubset(prices.index):
                    return prices

    # --- yfinance cached path (demo / fallback) ---
    # Use _v2 suffix to invalidate any pre-fix cache with pence prices
    cache_path = DATA_DIR / "current_prices_v2.parquet"
    if cache_path.exists():
        age_s = pd.Timestamp.now().timestamp() - cache_path.stat().st_mtime
        if age_s < 3600:
            cached = pd.read_parquet(cache_path)
            if set(tickers).issubset(cached.index):
                return cached.loc[tickers, "price"]

    ibkr_to_yf = {t: ibkr_to_yfinance(t, em.get(t, "")) for t in tickers}
    yf_to_ibkr = {v: k for k, v in ibkr_to_yf.items()}
    yf_tickers = list(dict.fromkeys(ibkr_to_yf.values()))

    data = yf.download(yf_tickers, period="5d", progress=False)["Close"]

    # Convert GBX (pence) → GBP (pounds) for LSE stocks (.L suffix)
    if isinstance(data, pd.Series):
        if any(data.name.endswith(s) for s in _GBP_PENCE_YF_SUFFIXES):
            data = data / 100.0
        ibkr_sym = yf_to_ibkr.get(data.name, data.name)
        prices = pd.Series({ibkr_sym: data.dropna().iloc[-1]}, name="price")
    else:
        for col in list(data.columns):
            if any(col.endswith(s) for s in _GBP_PENCE_YF_SUFFIXES):
                data[col] = data[col] / 100.0
        data = data.rename(columns=yf_to_ibkr)
        prices = data.ffill().iloc[-1].rename("price").dropna()

    prices.to_frame().to_parquet(cache_path)
    return prices


def fetch_historical_prices(tickers: list[str], period: str = "2y", exchange_map: dict[str, str] | None = None) -> pd.DataFrame:
    """Fetch daily close prices, cached to disk.

    Parameters
    ----------
    tickers : IBKR symbols
    period : yfinance period string, e.g. "2y"
    exchange_map : optional dict mapping IBKR symbol → IBKR exchange.
        Used to derive the correct yfinance ticker via ibkr_to_yfinance().

    Note: pence → pound conversion is NOT applied here because pct_change()
    returns are scale-invariant. The absolute price scale does not affect
    daily return calculations used in risk analysis.
    """
    key = "_".join(sorted(tickers)) + f"_{period}"
    cache_path = DATA_DIR / f"hist_{hash(key) & 0xFFFFFFFF:08x}.parquet"
    if cache_path.exists():
        age_s = pd.Timestamp.now().timestamp() - cache_path.stat().st_mtime
        if age_s < 86400:  # 24h cache
            return pd.read_parquet(cache_path)

    em = exchange_map or {}
    ibkr_to_yf = {t: ibkr_to_yfinance(t, em.get(t, "")) for t in tickers}
    yf_to_ibkr = {v: k for k, v in ibkr_to_yf.items()}
    yf_tickers = list(dict.fromkeys(ibkr_to_yf.values()))

    data = yf.download(yf_tickers, period=period, progress=False)["Close"]
    if isinstance(data, pd.Series):
        ibkr_sym = yf_to_ibkr.get(data.name, data.name)
        data = data.to_frame(ibkr_sym)
    else:
        data = data.rename(columns=yf_to_ibkr)
    data = data.ffill().dropna(how="all")
    data.to_parquet(cache_path)
    return data

print("Helpers defined ✓")


Helpers defined ✓


## 2. Portfolio Snapshot

In [4]:

# --- Build positions DataFrame (live or demo) ---

if LIVE_MODE:
    positions_df = await ib.positions.get_async()
    if positions_df.empty:
        display(HTML(
            '<div style="padding:10px;background:#3d3d0a;border-left:4px solid #ffd93d;color:#ffd93d">'
            '<b>No positions found</b> — falling back to demo portfolio.</div>'
        ))
        LIVE_MODE = False

if LIVE_MODE:
    # Filter to equities
    equity_positions = positions_df[positions_df["asset_class"] == "STK"].copy()
    other_positions = positions_df[positions_df["asset_class"] != "STK"]

    # Aggregate across accounts: sum shares, weighted-average cost basis; keep first contract_id
    equity_positions["total_cost"] = equity_positions["quantity"] * equity_positions["avg_cost"]
    equity_positions = (
        equity_positions.groupby("ticker", as_index=False)
        .agg(quantity=("quantity", "sum"), total_cost=("total_cost", "sum"),
             exchange=("exchange", "first"), asset_class=("asset_class", "first"),
             currency=("currency", "first"), contract_id=("contract_id", "first"))
    )
    equity_positions["avg_cost"] = equity_positions["total_cost"] / equity_positions["quantity"]
    equity_positions.drop(columns=["total_cost"], inplace=True)
    tickers = equity_positions["ticker"].tolist()

    # Account summary — use async version to avoid RuntimeError in Jupyter's running event loop
    summary = await ib.accounts.get_summary_async()
    net_liq_row = summary[summary["metric"] == "NetLiquidation"]
    cash_row = summary[summary["metric"] == "TotalCashValue"]
    net_liquidation = float(net_liq_row["amount"].iloc[0]) if not net_liq_row.empty else 0
    total_cash = float(cash_row["amount"].iloc[0]) if not cash_row.empty else 0

if not LIVE_MODE:
    # Demo mode
    rows = []
    for ticker, info in DEMO_PORTFOLIO.items():
        rows.append({
            "ticker": ticker,
            "quantity": info["shares"],
            "avg_cost": info["avg_cost"],
            "exchange": info["exchange"],
            "asset_class": "STK",
            "currency": "USD",
            "contract_id": None,
        })
    equity_positions = pd.DataFrame(rows)
    other_positions = pd.DataFrame()
    tickers = equity_positions["ticker"].tolist()
    total_cash = 25_000.0  # demo cash
    net_liquidation = 0  # will be computed below

# Build exchange map for IBKR → yfinance ticker translation (demo / fallback only)
exchange_map = dict(zip(equity_positions["ticker"], equity_positions["exchange"]))

# Build conid map so the IBKR snapshot path can look up prices by contract_id
conid_map = (
    dict(zip(equity_positions["ticker"], equity_positions["contract_id"]))
    if LIVE_MODE and "contract_id" in equity_positions.columns
    else None
)

# Fetch current market prices (LSE pence → GBP conversion happens inside fetch_current_prices):
#   LIVE_MODE  → IBKR snapshot (prices now in GBP for UK stocks, not pence)
#   demo/fallback → yfinance (same pence fix applied)
prices = await fetch_current_prices(
    list(dict.fromkeys(tickers)),
    exchange_map=exchange_map,
    ib=ib if LIVE_MODE else None,
    conid_map=conid_map,
)
prices_dict = prices if isinstance(prices, dict) else prices.to_dict()
equity_positions = equity_positions.reset_index(drop=True)
equity_positions["last_price"] = equity_positions["ticker"].map(prices_dict)
# market_value is in local currency (GBP for UK stocks, USD for US, HKD for HK, AUD for AU, etc.)
equity_positions["market_value"] = equity_positions["quantity"] * equity_positions["last_price"]

# --- FX-normalise to USD for cross-currency weight calculation ---
# Fetch spot FX rates for non-USD currencies present in the portfolio
_FX_PAIR_MAP = {"GBP": "GBPUSD=X", "HKD": "HKDUSD=X", "EUR": "EURUSD=X", "AUD": "AUDUSD=X"}
_port_currencies = [c for c in equity_positions["currency"].unique() if c != "USD" and c in _FX_PAIR_MAP]
_fx_rate_map = {"USD": 1.0}
if _port_currencies:
    _fx_pairs = [_FX_PAIR_MAP[c] for c in _port_currencies]
    _fx_raw = yf.download(_fx_pairs, period="2d", progress=False)["Close"]
    if isinstance(_fx_raw, pd.Series):
        _fx_raw = _fx_raw.to_frame(_fx_pairs[0])
    _fx_spot = _fx_raw.ffill().iloc[-1]
    for ccy in _port_currencies:
        pair = _FX_PAIR_MAP[ccy]
        if pair in _fx_spot.index and pd.notna(_fx_spot[pair]):
            _fx_rate_map[ccy] = float(_fx_spot[pair])

equity_positions["fx_rate"] = equity_positions["currency"].map(_fx_rate_map).fillna(1.0)
equity_positions["market_value_usd"] = equity_positions["market_value"] * equity_positions["fx_rate"]

# All downstream calculations use USD-normalised values so weights are comparable across currencies
# Use gross exposure as denominator so long-short portfolios
# do not get leveraged weights (near-zero net -> huge weights).
# For a long-only book gross == net, so this is backward-compatible.
gross_equity = equity_positions["market_value_usd"].abs().sum()
net_equity   = equity_positions["market_value_usd"].sum()
if not LIVE_MODE:
    net_liquidation = net_equity + total_cash

equity_positions["cost_basis"] = equity_positions["quantity"] * equity_positions["avg_cost"]
total_cost_basis = equity_positions["cost_basis"].sum()
# weight: positive for longs, negative for shorts; |weights| sum to 1
equity_positions["weight"] = equity_positions["market_value_usd"] / gross_equity

# Concentration metrics
hhi = (equity_positions["weight"] ** 2).sum()
top5_weight = equity_positions.nlargest(5, "weight")["weight"].sum()

price_source = "IBKR snapshot" if LIVE_MODE else "yfinance (demo)"
print(f"Portfolio: {len(tickers)} equity positions | NAV: ${net_liquidation:,.0f} | Cash: ${total_cash:,.0f}")
print(f"Price source: {price_source}")
print(f"HHI: {hhi:.4f} | Top-5 weight: {top5_weight:.1%}")
display(
    equity_positions[["ticker", "currency", "quantity", "avg_cost", "last_price", "market_value", "market_value_usd", "weight"]]
    .sort_values("weight", ascending=False)
    .style.format({
        "avg_cost": "{:,.2f}", "last_price": "{:,.2f}",
        "market_value": "{:,.0f}", "market_value_usd": "${:,.0f}", "weight": "{:.1%}",
    })
)


Portfolio: 33 equity positions | NAV: $98,943 | Cash: $16,724
Price source: IBKR snapshot
HHI: 0.0508 | Top-5 weight: 37.6%


,ticker,currency,quantity,avg_cost,last_price,market_value,market_value_usd,weight
30,SGLN,GBP,159.000000,56.47,64.00,"10,176","$13,645",12.4%
5,BARC,GBP,1500.000000,4.48,3.84,"5,761","$7,724",7.0%
13,GOOGL,USD,24.000000,312.14,290.44,"6,971","$6,971",6.3%
32,XOM,USD,40.000000,149.84,165.38,"6,615","$6,615",6.0%
17,KGC,USD,230.000000,24.28,27.92,"6,422","$6,422",5.8%
14,HAL,USD,160.000000,32.88,38.11,"6,098","$6,098",5.5%
23,NBIS,USD,50.000000,84.48,114.91,"5,746","$5,746",5.2%
2,AMZN,USD,25.000000,219.01,207.24,"5,181","$5,181",4.7%
27,PLS,AUD,1494.000000,1.74,4.54,"6,783","$4,740",4.3%
26,OKLO,USD,85.000000,67.02,54.96,"4,672","$4,672",4.2%


In [5]:
# --- Portfolio weights bar chart ---

sorted_pos = equity_positions.sort_values("weight", ascending=True)

fig_weights = go.Figure(go.Bar(
    x=sorted_pos["weight"],
    y=sorted_pos["ticker"],
    orientation="h",
    marker_color=COLORS["primary"],
    text=[f"{w:.1%}" for w in sorted_pos["weight"]],
    textposition="outside",
))
fig_weights.update_layout(
    **PLOTLY_LAYOUT,
    title="Portfolio Weights",
    xaxis=dict(tickformat=".0%", title="Weight"),
    yaxis=dict(title=""),
    margin=dict(l=60, r=80, t=40, b=40),
    height=max(300, len(tickers) * 35),
)
fig_weights.show()

## 3. Portfolio Risk Profile

Compute portfolio risk characteristics to inform the hedging strategy: beta, volatility, VaR/CVaR, drawdown, and correlation with VIX.

In [6]:
# --- Fetch 2-year historical prices for portfolio + SPY + VIX ---

all_tickers = tickers + ["SPY", "^VIX"]
hist_prices = fetch_historical_prices(all_tickers, period="2y", exchange_map=exchange_map)

# Daily returns — restrict to tickers that actually have history
available_tickers = [t for t in tickers if t in hist_prices.columns]
_raw_rets = hist_prices[available_tickers].pct_change()

# Winsorise extreme daily returns — yfinance adjusted-close data can contain
# bad points around reverse splits, M&A, or data errors. A move beyond +-50%
# in a single day is almost certainly a data artefact, not a real return.
MAX_DAILY_RETURN = 0.50
_extreme = _raw_rets.abs() > MAX_DAILY_RETURN
_n_clipped = int(_extreme.sum().sum())
if _n_clipped > 0:
    _clipped_info = _raw_rets[_extreme].stack().sort_values()
    print(f"WARNING: clipping {_n_clipped} extreme daily return(s) to +/-{MAX_DAILY_RETURN:.0%} "
          "(likely yfinance data error or unadjusted corporate action):")
    print(_clipped_info.map("{:.1%}".format).to_string())
    _raw_rets = _raw_rets.clip(lower=-MAX_DAILY_RETURN, upper=MAX_DAILY_RETURN)
# Keep only assets with at least 60 non-NaN observations so the covariance
# matrix is always full-rank (prevents rank-deficiency when recently-IPO'd
# or delisted stocks collapse the common window via dropna(how='any')).
MIN_HIST_OBS = 60
valid_ret_tickers = [c for c in _raw_rets.columns if _raw_rets[c].dropna().shape[0] >= MIN_HIST_OBS]
asset_returns = _raw_rets[valid_ret_tickers].dropna()
spy_returns = hist_prices["SPY"].pct_change().dropna()

# Portfolio weighted returns
weights_dict = dict(zip(equity_positions["ticker"], equity_positions["weight"]))
# Normalise using gross (absolute) weight sum so shorts are handled correctly.
# For long-only portfolios abs_sum == 1.0; for long-short it re-normalises to
# unit gross exposure.
equity_weight_sum = sum(abs(w) for w in weights_dict.values())
norm_weights = {t: w / equity_weight_sum for t, w in weights_dict.items()}

common_idx = asset_returns.index
port_returns = sum(
    asset_returns[t] * norm_weights.get(t, 0) for t in valid_ret_tickers if t in asset_returns.columns
)
port_returns.name = "Portfolio"

# VIX level series
vix_level = hist_prices["^VIX"].dropna()
vix_returns = vix_level.pct_change().dropna()

if len(common_idx) > 0:
    print(f"Historical data: {len(common_idx)} trading days ({common_idx[0].date()} → {common_idx[-1].date()})")
    excluded_from_rets = sorted(set(tickers) - set(valid_ret_tickers))
    if excluded_from_rets:
        print(f"  ⚠ Excluded from returns/risk analysis (< {MIN_HIST_OBS} obs): {', '.join(excluded_from_rets)}")
else:
    print("⚠ No historical data returned — check ticker mapping and exchange info.")

Date        Ticker
2026-01-27  SGLN       -99.0%
2024-07-24  ACG        -64.5%
2024-05-10  OKLO       -53.6%
2024-11-15  BE          59.2%
2025-09-24  LAC         95.8%
2026-03-24  SGLN      9996.2%
Historical data: 206 trading days (2025-06-06 → 2026-03-24)


In [7]:
# --- Diagnostics: identify outlier returns ---
print("Per-asset max single-day |return| (top 10 culprits):")
print(_raw_rets.abs().max().sort_values(ascending=False).head(10).map("{:.2%}".format))

print("Portfolio return distribution:")
print(port_returns.describe().map(lambda x: f"{x:.4%}"))

extreme_mask = port_returns.abs() > 0.50
if extreme_mask.any():
    print(f"WARNING: {extreme_mask.sum()} extreme portfolio-return day(s) (>+/-50%):")
    print(port_returns[extreme_mask].sort_values().map("{:.1%}".format))
    worst_day = port_returns.abs().idxmax()
    print(f"Worst day: {worst_day.date()} -- per-asset returns on that day:")
    print(asset_returns.loc[worst_day].sort_values().map("{:.1%}".format))
else:
    print("No extreme portfolio-return days found (all within +/-50%). Looks clean.")

Per-asset max single-day |return| (top 10 culprits):
Ticker
ACG     50.00%
SGLN    50.00%
BE      50.00%
OKLO    50.00%
LAC     50.00%
NBIS    49.42%
CRCL    35.47%
LEU     33.13%
DRO     31.40%
LAR     30.60%
dtype: object
Portfolio return distribution:
count    20600.0000%
mean         0.3156%
std          1.5492%
min         -4.7365%
25%         -0.4750%
50%          0.3158%
75%          1.1251%
max          6.5273%
Name: Portfolio, dtype: object
No extreme portfolio-return days found (all within +/-50%). Looks clean.


In [8]:
# --- Portfolio beta, volatility, VaR, drawdown ---

portfolio_beta = estimate_portfolio_beta(port_returns, spy_returns)
annual_vol = port_returns.std() * np.sqrt(252)
var_95 = np.percentile(port_returns.dropna(), 5)
cvar_95 = port_returns[port_returns <= var_95].mean()

# Drawdown
underwater = compute_underwater(port_returns)
current_dd = underwater.iloc[-1] if len(underwater) > 0 else 0
top_dd = compute_top_drawdowns(port_returns, top_n=5)

# Risk metrics table (reuse dashboard_v2)
metrics_table = compute_metrics_table(port_returns, spy_returns)

# --- Summary risk card ---
display(HTML(f"""
<div style="display:flex;gap:20px;flex-wrap:wrap;margin:10px 0">
  <div style="padding:15px;background:#1a1a2e;border-radius:8px;min-width:140px;text-align:center">
    <div style="color:#888;font-size:12px">Net Liquidation</div>
    <div style="color:#00d4ff;font-size:22px;font-weight:bold">${net_liquidation:,.0f}</div>
  </div>
  <div style="padding:15px;background:#1a1a2e;border-radius:8px;min-width:140px;text-align:center">
    <div style="color:#888;font-size:12px">Portfolio Beta</div>
    <div style="color:#ffd93d;font-size:22px;font-weight:bold">{portfolio_beta:.2f}</div>
  </div>
  <div style="padding:15px;background:#1a1a2e;border-radius:8px;min-width:140px;text-align:center">
    <div style="color:#888;font-size:12px">Annual Volatility</div>
    <div style="color:#bb86fc;font-size:22px;font-weight:bold">{annual_vol:.1%}</div>
  </div>
  <div style="padding:15px;background:#1a1a2e;border-radius:8px;min-width:140px;text-align:center">
    <div style="color:#888;font-size:12px">VaR (95% daily)</div>
    <div style="color:#ff6b6b;font-size:22px;font-weight:bold">{var_95:.2%}</div>
  </div>
  <div style="padding:15px;background:#1a1a2e;border-radius:8px;min-width:140px;text-align:center">
    <div style="color:#888;font-size:12px">Current Drawdown</div>
    <div style="color:#ff6b6b;font-size:22px;font-weight:bold">{current_dd:.1%}</div>
  </div>
</div>
"""))

display(metrics_table)

,Portfolio,SPY
Annual Return,114.70%,13.15%
Annual Volatility,24.59%,16.26%
Sharpe Ratio,4.481,0.532
Sortino Ratio,6.530,0.672
Calmar Ratio,7.977,0.701
Max Drawdown,-14.38%,-18.76%
VaR (95% daily),-2.36%,-1.53%
CVaR (95% daily),-3.18%,-2.36%
Win Rate (daily),64.08%,55.34%
Best Day,6.53%,10.50%


In [9]:
# --- Correlation heatmap (portfolio tickers + SPY + VIX) ---

corr_tickers = [t for t in tickers if t in hist_prices.columns] + ["SPY", "^VIX"]
corr_returns = hist_prices[corr_tickers].pct_change().dropna()
corr_matrix = compute_correlation_matrix(corr_returns)

fig_corr = go.Figure(go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.index,
    colorscale="RdBu_r",
    zmid=0,
    zmin=-1, zmax=1,
    text=[[f"{v:.2f}" for v in row] for row in corr_matrix.values],
    texttemplate="%{text}",
    textfont={"size": 10},
))
fig_corr.update_layout(
    **PLOTLY_LAYOUT,
    title="Correlation Matrix (incl. SPY & VIX)",
    margin=dict(l=60, r=20, t=40, b=60),
    height=500,
    yaxis=dict(autorange="reversed"),
)
fig_corr.show()

# Highlight VIX correlation
vix_corr = corr_matrix["^VIX"].drop("^VIX")
print(f"\nVIX correlation with portfolio holdings (negative = good hedge):")
for t, c in vix_corr.sort_values().items():
    print(f"  {t:>6s}: {c:+.3f}")


VIX correlation with portfolio holdings (negative = good hedge):
     SPY: -0.854
    AMZN: -0.484
   GOOGL: -0.431
      BE: -0.378
    VUSA: -0.359
    BARC: -0.342
    EQQQ: -0.341
    NBIS: -0.312
    OKLO: -0.286
    BIDU: -0.285
    CRCL: -0.269
    BABA: -0.269
     ARG: -0.269
    HY9H: -0.267
     LEU: -0.253
     HAL: -0.240
     LAR: -0.230
     HGT: -0.221
     ENR: -0.214
    NNND: -0.170
     RR.: -0.165
    PMET: -0.154
     LAC: -0.153
    1810: -0.152
     KGC: -0.094
     NTR: -0.080
     DRO: -0.042
     XOM: -0.030
     ACG: -0.003
    SGLN: +0.024
     PLS: +0.025
     LTR: +0.027
     LMT: +0.064
      CF: +0.109


In [10]:
# --- Underwater (drawdown) chart ---

fig_dd = go.Figure()
fig_dd.add_trace(go.Scatter(
    x=underwater.index, y=underwater.values,
    fill="tozeroy", fillcolor="rgba(255,107,107,0.15)",
    line=dict(color=COLORS["danger"], width=1),
    name="Drawdown",
))
fig_dd.add_hline(y=-0.10, line_dash="dash", line_color=COLORS["warning"],
                 annotation_text="-10%", annotation_position="bottom left")
fig_dd.add_hline(y=-0.20, line_dash="dash", line_color=COLORS["danger"],
                 annotation_text="-20%", annotation_position="bottom left")
fig_dd.update_layout(
    **PLOTLY_LAYOUT,
    title="Portfolio Underwater Chart",
    yaxis=dict(tickformat=".0%", title="Drawdown"),
    xaxis=dict(title=""),
    margin=dict(l=60, r=20, t=40, b=40),
    height=350,
)
fig_dd.show()

# Top drawdown periods
if not top_dd.empty:
    display(top_dd.style.format({
        "depth": "{:.1%}",
        "duration_days": "{:.0f}",
        "recovery_days": "{:.0f}",
    }))

,start,trough,end,depth,duration_days,recovery_days
0,2026-01-26 00:00:00,2026-02-05 00:00:00,2026-03-24 00:00:00,-14.4%,57,nan
1,2025-10-27 00:00:00,2025-11-21 00:00:00,2025-12-11 00:00:00,-8.7%,45,20
2,2025-10-16 00:00:00,2025-10-22 00:00:00,2025-10-24 00:00:00,-5.6%,8,2
3,2025-12-12 00:00:00,2025-12-17 00:00:00,2025-12-22 00:00:00,-4.7%,10,5
4,2025-08-13 00:00:00,2025-08-20 00:00:00,2025-08-27 00:00:00,-3.5%,14,7


In [11]:
# --- Risk contribution per holding ---

# Filter to tickers that have BOTH price history AND a current portfolio weight.
# Without this, tickers with history but no weight (e.g. closed positions) get
# weight=0, and after normalisation inside compute_risk_contribution, any single
# ticker with a real weight ends up at 100% — a degenerate result.
avail_tickers = [
    t for t in valid_ret_tickers
    if t in asset_returns.columns and norm_weights.get(t, 0) > 0
]
rc = compute_risk_contribution(asset_returns[avail_tickers], norm_weights)
rc_sorted = rc.sort_values("pct_contribution", ascending=True)

fig_rc = go.Figure(go.Bar(
    x=rc_sorted["pct_contribution"],
    y=rc_sorted["asset"],
    orientation="h",
    marker_color=[COLORS["danger"] if v > 0.15 else COLORS["primary"] for v in rc_sorted["pct_contribution"]],
    text=[f"{v:.1%}" for v in rc_sorted["pct_contribution"]],
    textposition="outside",
))
fig_rc.update_layout(
    **PLOTLY_LAYOUT,
    title="Risk Contribution by Holding",
    xaxis=dict(tickformat=".0%", title="% of Portfolio Risk"),
    yaxis=dict(title=""),
    margin=dict(l=60, r=80, t=40, b=40),
    height=max(300, len(avail_tickers) * 35),
)
fig_rc.show()

## 4. VIX Options Chain — Data Acquisition

**Important:** VIX options are **European-style** and **cash-settled**. They settle against the **Special Opening Quotation (SOQ)** of the VIX index, *not* the spot VIX. The SOQ is calculated from the opening prices of S&P 500 options on settlement morning and can differ significantly from spot VIX. Data below is delayed ~15 minutes (free sources).

In [ ]:
import asyncio

# --- VIX section shared state ---
# Declared here so the source-selector, chain display, and analytics cells
# can all reference the same objects.

_chain_cache: dict[str, pd.DataFrame] = {}
expirations: list[str] = []
vix_data = None

expiry_dropdown = widgets.Dropdown(
    options=["(no data)"],
    description="Expiry:",
    style={"description_width": "60px"},
)
chain_output = widgets.Output()

# w_expiry is also used in section 5 analytics; pre-declared here so
# _init_vix_data() can refresh it when the source changes.
w_expiry = widgets.Dropdown(
    options=["(no data)"],
    description="Expiry:",
)


In [ ]:

# --- VIX data source selector ---
# yfinance  -- no auth, ~15 min delay, no Greeks
# cboe      -- no auth, ~15 min delay, includes Greeks + OI (recommended)
# tradier   -- free signup at developer.tradier.com, includes Greeks

_SOURCE_LABELS = {
    "CBOE Direct (no auth, recommended)": "cboe",
    "yfinance (no auth)": "yfinance",
    "Tradier Sandbox (free signup, includes Greeks)": "tradier",
}

w_source = widgets.RadioButtons(
    options=list(_SOURCE_LABELS.keys()),
    value="CBOE Direct (no auth, recommended)",
    description="Data source:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="520px"),
)
w_tradier_token = widgets.Text(
    placeholder="Paste your Tradier sandbox token here",
    description="Token:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="460px", display="none"),
)
w_load_btn = widgets.Button(
    description="Load expirations",
    button_style="primary",
    icon="refresh",
)
source_status = widgets.Output()

# Current VIX spot (computed from historical data fetched in section 3)
current_vix = float(vix_level.iloc[-1]) if len(vix_level) > 0 else 20.0


async def _init_vix_data():
    global vix_data, expirations
    source_key = _SOURCE_LABELS[w_source.value]
    token = w_tradier_token.value.strip() or None

    source_status.clear_output(wait=True)
    with source_status:
        display(HTML(
            '<div style="color:#ffd93d;padding:4px 0">Connecting to '
            f'<b>{w_source.value}</b>...</div>'
        ))

    try:
        if LIVE_MODE and ib is not None:
            vix_data = VIXData(options=ib.options)
        else:
            vix_data = VIXData(source=source_key, tradier_token=token)

        exps = await vix_data.get_expirations_async()
        expirations = exps

        source_status.clear_output(wait=True)
        with source_status:
            display(HTML(
                '<div style="padding:8px 12px;background:#0a3d0a;border-left:4px solid #00ff88;'
                'color:#00ff88;border-radius:4px">'
                f'<b>{w_source.value}</b> -- {len(expirations)} expirations loaded. '
                f'VIX spot: <b>{current_vix:.2f}</b>'
                '</div>'
            ))
            print(f"First expirations: {', '.join(expirations[:8])}{'...' if len(expirations) > 8 else ''}")

        # Refresh downstream dropdowns and clear cached chains
        _chain_cache.clear()
        opts = expirations[:12] if expirations else ["(no data)"]
        expiry_dropdown.options = opts
        if opts:
            expiry_dropdown.value = opts[0]
        w_expiry.options = opts
        if opts:
            w_expiry.value = opts[0]

    except Exception as exc:
        source_status.clear_output(wait=True)
        with source_status:
            display(HTML(
                '<div style="padding:8px 12px;background:#3d0a0a;border-left:4px solid #ff6b6b;'
                'color:#ff6b6b;border-radius:4px">'
                f'<b>Error:</b> {exc}</div>'
            ))


def _on_source_change(change):
    src = _SOURCE_LABELS[change["new"]]
    w_tradier_token.layout.display = "flex" if src == "tradier" else "none"


def _on_load_btn_clicked(_):
    asyncio.ensure_future(_init_vix_data())


w_source.observe(_on_source_change, names="value")
w_load_btn.on_click(_on_load_btn_clicked)

display(widgets.VBox([w_source, w_tradier_token, w_load_btn, source_status]))

# Initial load -- awaited so downstream cells have expirations when they run
await _init_vix_data()


In [ ]:
# --- Interactive expiry selector -> VIX calls table ---
# _chain_cache, expiry_dropdown, and chain_output are declared in the setup cell above.


async def _load_chain(exp: str) -> None:
    with chain_output:
        if exp in ("(no data)", None, ""):
            print("No VIX expiration data available.")
            return
        try:
            if exp not in _chain_cache:
                _chain_cache[exp] = await vix_data.get_calls_async(exp, net_liquidation)
            calls = _chain_cache[exp]
            if calls.empty:
                print(f"No call data for expiry {exp}")
                return

            display_cols = [c for c in [
                "strike", "bid", "ask", "mid", "last", "volume",
                "open_interest", "implied_vol", "days_to_expiry",
                "moneyness", "cost_per_contract", "cost_bps", "breakeven_vix",
            ] if c in calls.columns]

            styled = calls[display_cols].style.format({
                "strike": "{:.1f}", "bid": "{:.2f}", "ask": "{:.2f}",
                "mid": "{:.2f}", "last": "{:.2f}",
                "implied_vol": "{:.2%}", "moneyness": "{:+.1%}",
                "cost_per_contract": "${:,.0f}", "cost_bps": "{:.1f}",
                "breakeven_vix": "{:.1f}",
            }, na_rep="--")
            display(styled)
        except Exception as e:
            print(f"Error fetching chain: {e}")


def on_expiry_change(change):
    chain_output.clear_output(wait=True)
    asyncio.ensure_future(_load_chain(change["new"]))


expiry_dropdown.observe(on_expiry_change, names="value")
display(expiry_dropdown, chain_output)

# Trigger initial load if expirations were fetched above
if expiry_dropdown.value not in ("(no data)", None, ""):
    asyncio.ensure_future(_load_chain(expiry_dropdown.value))


Dropdown(description='Expiry:', options=('20260324', '20260331', '20260407', '20260414', '20260421', '20260518…

Output(msg_id='3a2e4862-b06f-4dcc-8940-508f9fe7c7cf')

In [17]:
# --- VIX options term structure (ATM call price by expiry) ---

try:
    term_struct = await vix_data.get_term_structure_async(net_liquidation)
    if not term_struct.empty:
        fig_ts = go.Figure()
        fig_ts.add_trace(go.Scatter(
            x=term_struct["days_to_expiry"],
            y=pd.to_numeric(term_struct["atm_mid"], errors="coerce"),
            mode="lines+markers",
            line=dict(color=COLORS["primary"], width=2),
            marker=dict(size=8),
            name="ATM Call Mid",
        ))
        fig_ts.update_layout(
            **PLOTLY_LAYOUT,
            title="VIX Options Term Structure (ATM Call Price)",
            xaxis=dict(title="Days to Expiry"),
            yaxis=dict(title="ATM Call Mid Price ($)"),
            margin=dict(l=60, r=20, t=40, b=40),
            height=350,
        )
        fig_ts.show()
    else:
        print("Term structure data not available.")
except Exception as e:
    print(f"Could not build term structure: {e}")


Unknown contract: Option(symbol='VIX', lastTradeDateOrContractMonth='20260616', strike=21.5, right='C', exchange='SMART', tradingClass='VIX')
Unknown contract: Option(symbol='VIX', lastTradeDateOrContractMonth='20260616', strike=21.5, right='P', exchange='SMART', tradingClass='VIX')
Unknown contract: Option(symbol='VIX', lastTradeDateOrContractMonth='20260616', strike=22.5, right='C', exchange='SMART', tradingClass='VIX')
Unknown contract: Option(symbol='VIX', lastTradeDateOrContractMonth='20260616', strike=22.5, right='P', exchange='SMART', tradingClass='VIX')
Unknown contract: Option(symbol='VIX', lastTradeDateOrContractMonth='20260616', strike=23.5, right='C', exchange='SMART', tradingClass='VIX')
Unknown contract: Option(symbol='VIX', lastTradeDateOrContractMonth='20260616', strike=23.5, right='P', exchange='SMART', tradingClass='VIX')
Unknown contract: Option(symbol='VIX', lastTradeDateOrContractMonth='20260616', strike=24.5, right='C', exchange='SMART', tradingClass='VIX')
Unknow

## 5. VIX Call Option Analytics & Comparison

For each candidate VIX call, compute: cost of protection, breakeven, hedge ratio, and cost-efficiency. Use the widgets to filter and compare options.

In [ ]:
# --- Interactive analytics: expiry + strike range + target protection ---

# Instantiate scenario engine for contracts_needed calculation
engine = ScenarioEngine(net_liquidation, portfolio_beta, current_vix)

# w_expiry is shared with the source-selector (declared in the section 4 setup cell).
# Its options are updated automatically when a new source is loaded.
w_target = widgets.Dropdown(
    options=[("10% drawdown", 0.10), ("20% drawdown", 0.20), ("30% drawdown", 0.30),
             ("40% drawdown", 0.40), ("50% drawdown", 0.50)],
    value=0.30,
    description="Protect vs:",
    style={"description_width": "80px"},
)
w_contracts = widgets.IntSlider(
    value=5, min=1, max=50, step=1, description="Contracts:",
    style={"description_width": "80px"},
)

analytics_output = widgets.Output()


def update_analytics(*_):
    analytics_output.clear_output(wait=True)
    with analytics_output:
        exp = w_expiry.value
        if not exp or exp == "(no data)":
            print("No data available.")
            return

        try:
            if exp not in _chain_cache:
                _chain_cache[exp] = vix_data.get_calls(exp, net_liquidation)
            calls = _chain_cache[exp]
            if calls.empty:
                print(f"No calls for {exp}")
                return
        except Exception as e:
            print(f"Error: {e}")
            return

        target_dd = w_target.value
        n_contracts = w_contracts.value

        # Add contracts needed for target
        df = calls.copy()
        df["contracts_needed"] = df["strike"].apply(
            lambda k: engine.contracts_needed(0.5, k, -target_dd)
        )
        mid_vals = pd.to_numeric(df["mid"], errors="coerce")
        df["total_cost"] = mid_vals * VIX_MULTIPLIER * n_contracts
        df["total_cost_bps"] = np.where(
            net_liquidation > 0,
            df["total_cost"] / net_liquidation * 10_000,
            0,
        )

        # Display top options ranked by payoff_ratio_40
        show_cols = [c for c in [
            "strike", "mid", "cost_bps", "breakeven_vix",
            "payoff_ratio_40", "contracts_needed", "total_cost", "total_cost_bps",
            "payoff_at_40", "payoff_at_60", "days_to_expiry",
        ] if c in df.columns]
        top = df.nlargest(15, "payoff_ratio_40")[show_cols]
        display(top.style.format({
            "strike": "{:.0f}", "mid": "{:.2f}", "cost_bps": "{:.1f}",
            "breakeven_vix": "{:.1f}", "payoff_ratio_40": "{:.1f}x",
            "total_cost": "${:,.0f}", "total_cost_bps": "{:.1f}",
            "payoff_at_40": "${:,.0f}", "payoff_at_60": "${:,.0f}",
        }, na_rep="--"))

        # --- Chart 1: Cost of protection vs strike ---
        fig1 = go.Figure(go.Bar(
            x=df["strike"], y=pd.to_numeric(df["cost_bps"], errors="coerce"),
            marker_color=COLORS["warning"],
            name="Cost (bps)",
        ))
        fig1.update_layout(
            **PLOTLY_LAYOUT,
            title=f"Cost of Protection vs Strike ({exp})",
            xaxis=dict(title="Strike"), yaxis=dict(title="Cost (bps of NAV)"),
            margin=dict(l=60, r=20, t=40, b=40), height=300,
        )
        fig1.show()

        # --- Chart 2: Payoff ratio at VIX=40,60,80 ---
        fig2 = go.Figure()
        for lvl, color in [(40, COLORS["success"]), (60, COLORS["warning"]), (80, COLORS["danger"])]:
            col = f"payoff_at_{lvl}"
            if col in df.columns:
                ratio = np.where(
                    pd.to_numeric(df["cost_per_contract"], errors="coerce") > 0,
                    pd.to_numeric(df[col], errors="coerce") / pd.to_numeric(df["cost_per_contract"], errors="coerce"),
                    0,
                )
                fig2.add_trace(go.Bar(x=df["strike"], y=ratio, name=f"VIX={lvl}", marker_color=color))
        fig2.update_layout(
            **PLOTLY_LAYOUT,
            title="Payoff Ratio by VIX Level",
            xaxis=dict(title="Strike"), yaxis=dict(title="Payoff / Premium"),
            barmode="group",
            margin=dict(l=60, r=20, t=40, b=40), height=350,
        )
        fig2.show()

        # --- Chart 3: Breakeven VIX vs strike ---
        fig3 = go.Figure()
        fig3.add_trace(go.Scatter(
            x=df["strike"],
            y=pd.to_numeric(df["breakeven_vix"], errors="coerce"),
            mode="lines+markers",
            line=dict(color=COLORS["primary"], width=2),
            name="Breakeven VIX",
        ))
        fig3.add_hline(y=current_vix, line_dash="dash", line_color=COLORS["gray"],
                       annotation_text=f"Current VIX: {current_vix:.1f}")
        fig3.update_layout(
            **PLOTLY_LAYOUT,
            title="Breakeven VIX Level vs Strike",
            xaxis=dict(title="Strike"), yaxis=dict(title="VIX Level"),
            margin=dict(l=60, r=20, t=40, b=40), height=300,
        )
        fig3.show()


w_expiry.observe(update_analytics, names="value")
w_target.observe(update_analytics, names="value")
w_contracts.observe(update_analytics, names="value")

display(widgets.HBox([w_expiry, w_target, w_contracts]))
display(analytics_output)

# Trigger initial
update_analytics()


## 6. Scenario Analysis — Drawdown Modelling

Model how the portfolio behaves under stress and how VIX calls provide insurance. Select a strike and number of contracts to see the impact across historical and hypothetical scenarios.

In [ ]:
# --- 6a. Historical stress scenarios table ---

# Pick a representative strike for initial display (ATM + ~5 points OTM)
default_strike = round(current_vix + 5)
default_premium = 2.50  # reasonable estimate; will be refined from chain data

# If we have chain data, use a real mid price
if _chain_cache:
    first_chain = next(iter(_chain_cache.values()))
    if not first_chain.empty:
        atm_row = first_chain.iloc[(first_chain["strike"] - default_strike).abs().argsort()[:1]]
        if not atm_row.empty:
            mid_val = pd.to_numeric(atm_row["mid"], errors="coerce").iloc[0]
            if pd.notna(mid_val) and mid_val > 0:
                default_premium = float(mid_val)
                default_strike = float(atm_row["strike"].iloc[0])

print(f"Default analysis: Strike = {default_strike:.0f}, Premium = ${default_premium:.2f}, VIX spot = {current_vix:.1f}")
print()

stress = engine.stress_table(default_strike, default_premium, 5)
display(stress.style.format({
    "spx_drawdown": "{:.0%}",
    "vix_start": "{:.0f}", "vix_peak": "{:.0f}",
    "portfolio_loss": "${:,.0f}", "hedge_gain": "${:,.0f}",
    "net_pnl": "${:,.0f}", "recovery_pct": "{:.0%}",
}))

In [ ]:
# --- 6b. Interactive drawdown model chart ---

# Build available strikes from chain data
available_strikes = sorted(set(
    float(row["strike"])
    for df in _chain_cache.values()
    for _, row in df.iterrows()
    if pd.notna(row.get("strike"))
))
if not available_strikes:
    available_strikes = [float(round(current_vix + i)) for i in range(-5, 25)]

# Build strike → premium mapping
strike_premium_map: dict[float, float] = {}
for df in _chain_cache.values():
    for _, row in df.iterrows():
        k = float(row["strike"])
        m = pd.to_numeric(row.get("mid"), errors="coerce")
        if pd.notna(m) and m > 0 and k not in strike_premium_map:
            strike_premium_map[k] = float(m)

w_strike = widgets.Dropdown(
    options=[(f"{k:.0f}", k) for k in available_strikes],
    value=min(available_strikes, key=lambda k: abs(k - default_strike)),
    description="Strike:",
)
w_num = widgets.IntSlider(value=5, min=1, max=50, step=1, description="Contracts:")
scenario_output = widgets.Output()


def update_scenarios(*_):
    scenario_output.clear_output(wait=True)
    with scenario_output:
        strike = w_strike.value
        n = w_num.value
        premium = strike_premium_map.get(strike, default_premium)

        # Drawdown curve
        curve = engine.drawdown_curve(strike, premium, n)

        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=curve["spx_drawdown"] * 100, y=curve["unhedged_value"],
            name="Unhedged", line=dict(color=COLORS["danger"], width=2),
        ))
        fig.add_trace(go.Scatter(
            x=curve["spx_drawdown"] * 100, y=curve["hedged_value"],
            name="Hedged", line=dict(color=COLORS["success"], width=2),
            fill="tonexty", fillcolor="rgba(0,255,136,0.1)",
        ))

        # VIX on secondary axis
        fig.add_trace(go.Scatter(
            x=curve["spx_drawdown"] * 100, y=curve["vix_estimate"],
            name="VIX Estimate", line=dict(color=COLORS["warning"], width=1.5, dash="dash"),
            yaxis="y2",
        ))

        fig.update_layout(
            **PLOTLY_LAYOUT,
            title=f"Hedged vs Unhedged Portfolio — Strike {strike:.0f}, {n} contracts @ ${premium:.2f}",
            xaxis=dict(title="S&P 500 Drawdown (%)", ticksuffix="%"),
            yaxis=dict(title="Portfolio Value ($)", tickformat="$,.0f"),
            yaxis2=dict(title="VIX Level", overlaying="y", side="right",
                        showgrid=False, tickformat=".0f"),
            legend=dict(orientation="h", yanchor="bottom", y=1.02),
            margin=dict(l=80, r=60, t=60, b=40),
            height=450,
        )
        fig.show()

        # Stress table for this strike
        stress = engine.stress_table(strike, premium, n)
        display(stress.style.format({
            "spx_drawdown": "{:.0%}",
            "vix_start": "{:.0f}", "vix_peak": "{:.0f}",
            "portfolio_loss": "${:,.0f}", "hedge_gain": "${:,.0f}",
            "net_pnl": "${:,.0f}", "recovery_pct": "{:.0%}",
        }))


w_strike.observe(update_scenarios, names="value")
w_num.observe(update_scenarios, names="value")

display(widgets.HBox([w_strike, w_num]))
display(scenario_output)
update_scenarios()

In [ ]:
# --- 6c. Payoff matrix heatmap ---

# Select subset of strikes for heatmap
heatmap_strikes = [k for k in available_strikes if current_vix - 5 <= k <= current_vix + 30]
if len(heatmap_strikes) > 20:
    step = max(1, len(heatmap_strikes) // 20)
    heatmap_strikes = heatmap_strikes[::step]

if heatmap_strikes:
    premiums_map = {k: strike_premium_map.get(k, default_premium) for k in heatmap_strikes}
    n_contracts = w_num.value if hasattr(w_num, "value") else 5

    matrix = engine.payoff_matrix(heatmap_strikes, premiums_map, n_contracts)

    fig_hm = go.Figure(go.Heatmap(
        z=matrix.values,
        x=matrix.columns,
        y=[f"{k:.0f}" for k in matrix.index],
        colorscale="RdYlGn",
        zmid=0,
        text=[[f"${v:,.0f}" for v in row] for row in matrix.values],
        texttemplate="%{text}",
        textfont={"size": 9},
    ))
    fig_hm.update_layout(
        **PLOTLY_LAYOUT,
        title=f"Net P&L Heatmap — {n_contracts} contracts (green = positive, red = negative)",
        xaxis=dict(title="S&P 500 Drawdown"),
        yaxis=dict(title="VIX Call Strike", autorange="reversed"),
        margin=dict(l=60, r=20, t=40, b=40),
        height=max(300, len(heatmap_strikes) * 30),
    )
    fig_hm.show()
else:
    print("No strike data available for heatmap.")

## 7. Recommendations & Rollover Strategy

Synthesise the analysis into actionable hedge recommendations across three risk profiles.

In [ ]:
# --- 7a. Optimal hedge recommendations ---

# Combine all cached chains for the advisor
all_calls = pd.concat(_chain_cache.values(), ignore_index=True) if _chain_cache else pd.DataFrame()

if not all_calls.empty:
    advisor = HedgeAdvisor(all_calls, engine, current_vix)
    recs = advisor.recommend_all()

    if not recs.empty:
        display(HTML("<h3>Hedge Recommendations by Profile</h3>"))
        display(recs.style.format({
            "strike": "{:.0f}", "mid": "{:.2f}",
            "cost_per_contract": "${:,.0f}", "cost_bps": "{:.1f}",
            "breakeven_vix": "{:.1f}", "contracts_needed": "{:.0f}",
            "total_cost": "${:,.0f}", "total_cost_bps": "{:.1f}",
            "payoff_ratio_40": "{:.1f}x",
            "net_payout_vix_40": "${:,.0f}", "net_payout_vix_60": "${:,.0f}",
        }, na_rep="—"))
    else:
        print("No options matched the hedge profile filters. Try during market hours for better data.")

    # Summary card for moderate recommendation
    card = advisor.summary_card()
    if card.get("available"):
        display(HTML(f"""
        <div style="margin:20px 0;padding:20px;background:#1a1a2e;border-left:4px solid {COLORS['success']};border-radius:8px">
            <h3 style="color:{COLORS['success']};margin-top:0">Recommended Hedge (Moderate Profile)</h3>
            <table style="color:#e0e0e0;font-size:14px">
                <tr><td style="padding:4px 20px 4px 0;color:#888">Contract</td>
                    <td>VIX {card['strike']:.0f}C — Expiry {card['expiry']}</td></tr>
                <tr><td style="padding:4px 20px 4px 0;color:#888">Contracts</td>
                    <td>{card['contracts']}</td></tr>
                <tr><td style="padding:4px 20px 4px 0;color:#888">Total Cost</td>
                    <td>${card['total_cost']:,.0f} ({card['total_cost_bps']:.1f} bps of NAV)</td></tr>
                <tr><td style="padding:4px 20px 4px 0;color:#888">Breakeven VIX</td>
                    <td>{card['breakeven_vix']:.1f} (current: {card['current_vix']:.1f})</td></tr>
                <tr><td style="padding:4px 20px 4px 0;color:#888">Days to Expiry</td>
                    <td>{card['days_to_expiry']}</td></tr>
            </table>
        </div>
        """))
else:
    print("No VIX call data available for recommendations.")

In [ ]:
# --- 7b. Rollover strategy & cost estimation ---

if not all_calls.empty and len(expirations) >= 2:
    try:
        ts = await vix_data.get_term_structure_async(net_liquidation)
        if len(ts) >= 2:
            near = ts.iloc[0]
            far = ts.iloc[1]
            near_mid = pd.to_numeric(near.get("atm_mid"), errors="coerce")
            far_mid = pd.to_numeric(far.get("atm_mid"), errors="coerce")
            days_btwn = int(far["days_to_expiry"]) - int(near["days_to_expiry"])

            if pd.notna(near_mid) and pd.notna(far_mid) and near_mid > 0 and far_mid > 0:
                roll = HedgeAdvisor.rollover_cost(
                    float(near_mid), float(far_mid), max(days_btwn, 1), net_liquidation,
                )
                contango = far_mid > near_mid

                display(HTML(f"""
                <div style="margin:20px 0;padding:20px;background:#1a1a2e;border-radius:8px">
                    <h3 style="color:{COLORS['primary']};margin-top:0">Rollover Cost Estimate</h3>
                    <p style="color:#ccc">
                        Near expiry: <b>{near['expiry']}</b> (ATM mid: ${float(near_mid):.2f}) →
                        Far expiry: <b>{far['expiry']}</b> (ATM mid: ${float(far_mid):.2f})<br>
                        Term structure: <b>{'Contango' if contango else 'Backwardation'}</b>
                        {'(rolling costs money — you pay more for the far expiry)' if contango else '(rolling earns a credit)'}<br><br>
                        Roll cost per contract: <b>${roll['roll_cost_per_contract']:,.0f}</b><br>
                        Annualised cost per contract: <b>${roll['annualized_cost_per_contract']:,.0f}</b><br>
                        Annualised cost: <b>{roll['annualized_cost_bps']:.1f} bps</b> of NAV per contract
                    </p>
                </div>
                """))

                # Term structure chart
                fig_roll = go.Figure()
                fig_roll.add_trace(go.Scatter(
                    x=ts["days_to_expiry"].astype(int),
                    y=pd.to_numeric(ts["atm_mid"], errors="coerce"),
                    mode="lines+markers",
                    line=dict(color=COLORS["primary"] if contango else COLORS["success"], width=2),
                    marker=dict(size=8),
                    text=[f"Exp: {e}" for e in ts["expiry"]],
                    name="ATM Call Mid",
                ))
                fig_roll.update_layout(
                    **PLOTLY_LAYOUT,
                    title=f"VIX Options Term Structure ({'Contango' if contango else 'Backwardation'})",
                    xaxis=dict(title="Days to Expiry"),
                    yaxis=dict(title="ATM Call Mid ($)"),
                    margin=dict(l=60, r=20, t=40, b=40),
                    height=300,
                )
                fig_roll.show()
            else:
                print("Insufficient ATM pricing data for rollover analysis.")
        else:
            print("Need at least 2 expirations for rollover analysis.")
    except Exception as e:
        print(f"Rollover analysis error: {e}")
else:
    print("Rollover analysis requires VIX options data with at least 2 expirations.")


In [ ]:
# --- 7c. Position risk attribution ---

display(HTML("<h3>Per-Holding Risk Attribution</h3>"))
display(HTML("<p style='color:#ccc'>Which holdings drive the most portfolio risk, and how VIX hedge offsets via negative correlation.</p>"))

rc_display = rc.copy()
# Add VIX correlation per holding
for i, row in rc_display.iterrows():
    t = row["asset"]
    if t in corr_matrix.index and "^VIX" in corr_matrix.columns:
        rc_display.loc[i, "vix_correlation"] = corr_matrix.loc[t, "^VIX"]
    else:
        rc_display.loc[i, "vix_correlation"] = np.nan

display(rc_display.sort_values("pct_contribution", ascending=False).style.format({
    "weight": "{:.1%}",
    "marginal_risk": "{:.4f}",
    "risk_contribution": "{:.4f}",
    "pct_contribution": "{:.1%}",
    "vix_correlation": "{:+.3f}",
}, na_rep="—"))

In [ ]:
# --- Cleanup: disconnect IBKR if connected ---

if LIVE_MODE and ib is not None:
    try:
        ib.disconnect()
        print("IBKR disconnected.")
    except Exception:
        pass

print("\nNotebook complete.")